# Is my dataset suited for QML?

Quantum models are not a free upgrade over classical ones, so before training them it is worth asking whether they have any chance on our dataset. Falcondale's assessment answers that in a few seconds, using only classical resources, and ranks the models we could train.

We will start with a toy dataset where classes are two concentric circles, something a linear model cannot separate.

In [1]:
import pandas as pd
from sklearn import datasets

X, y = datasets.make_circles(400, noise=0.08, factor=0.5, random_state=0)
circles = pd.DataFrame(X, columns=["x", "y"]).assign(target=y)
circles.head()

,x,y,target
0,-0.537188,-0.813996,0
1,-0.299402,-0.142876,1
2,-0.215971,-0.507825,1
3,0.955027,-0.086815,0
4,-0.060597,0.501118,1


As usual, we create a project and preprocess the data. Then `assess()` runs the assessment, which is displayed as a report in the notebook.

In [2]:
from falcondale import Project

myproject = Project(circles, target="target")
myproject.preprocess()

myproject.assess()

╭─ QML suitability ────────────────────────────────────────────────────────────╮
│ 89.8 / 100   HIGH  Quantum models are worth a try                            │
│ 400 of 400 samples, 2 of 2 features, 2 classes, budget of 16 qubits          │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Sub-scores ─────────────────────────────────────────────────────────────────╮
│ Fourier approx.     ━━━━━━━━━━━━━━━━━━━━━━    100  average R² 0.845          │
│ Trainability        ━━━━━━━━━━━━━━━━━━╺━━━     83  16.7% of frequencies      │
│                                                    unreachable (ternary      │
│                                                    init)                     │
│ Circuit depth       ━━━━━━━━━━━━━╺━━━━━━━━     60  fixed frequency depth ≈   │
│                                                    416                       │
│ Classical hardness  ━━━━━━━━━━━━━━━━━━━━━━    100  logistic bal. acc. 0.492, │
│                                                    tree gap +0.470           │
│ Class balance       ━━━━━━━━━━━━━━━━━━━━━━    100  ratio 1.000 (balanced)    │
│ Dimensionality      ━━━━━━━━━━━━━━━━━━━━━━    100  ≥ 3 qubits needed         │
╰──────────────────────────────────────────────────────────────────────────────╯
Raw table: 2 columns become 2 features                                          
┏━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Column   ┃ Kind       ┃   Missing ┃   Unique ┃   Features ┃    Target assoc. ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ x        │ numeric    │        0% │      400 │          1 │             0.62 │
│ y        │ numeric    │        0% │      400 │          1 │             0.60 │
└──────────┴────────────┴───────────┴──────────┴────────────┴──────────────────┘
Classical baseline (3 fold, out of fold)                                        
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Model                 ┃  Bal. accuracy ┃  Precision ┃ Recall ┃    F1 ┃   AUC ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ Logistic regression   │          0.492 │      0.493 │  0.510 │ 0.501 │ 0.472 │
│ Decision tree         │          0.962 │      0.965 │  0.960 │ 0.962 │ 0.963 │
└───────────────────────┴────────────────┴────────────┴────────┴───────┴───────┘
Spectrum                                  
┌─────────────────────────────┬──────────┐
│ Features showing signal     │   2 of 2 │
│ Highest frequency ω         │        4 │
│ Significant frequencies K   │        5 │
│ Unreachable, zero init      │    58.3% │
│ Unreachable, ternary init   │    16.7% │
├─────────────────────────────┼──────────┤
│ RBF kernel effective rank   │     2.62 │
│ RBF kernel spectral gap     │    0.394 │
│ RBF kernel condition number │ 2.04e+13 │
└─────────────────────────────┴──────────┘
Most oscillating features                                                       
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Feature            ┃     ω ┃     K ┃           R² ┃              Unreachable ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ x                  │     4 │     3 │        0.895 │                      1/3 │
│ y                  │     2 │     2 │        0.794 │                      0/2 │
└────────────────────┴───────┴───────┴──────────────┴──────────────────────────┘
Recommended models                                                              
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Model                    ┃ Qubits ┃ Score ┃ Why                          ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ qsvc (backend=qiskit)    │      2 │   100 │ 83% of the frequencies       │
│   │                          │        │       │ within reach of a repeated   │
│

The labels switch class only a few times along each feature, so most frequencies are within reach of a trainable encoding, and the logistic regression is no better than a coin flip while a decision tree does great. Everything points to a nonlinear kernel, and the ZZ feature map of the Qiskit QSVC comes first.

The project keeps the assessment, so `evaluate("auto")` trains the recommended model straight away. Let's compare it with the classical baseline.

In [3]:
model = myproject.evaluate("auto")
baseline = myproject.evaluate("svc")

print(f"{model.type}: {model.metric('auc'):.3f} AUC, svc: {baseline.metric('auc'):.3f} AUC")

Assessment recommends qsvc (backend=qiskit): 83% of the frequencies within reach of a repeated encoding


qsvc: 1.000 AUC, svc: 1.000 AUC


Both reach a perfect score, circles are easy for any nonlinear kernel after all. What matters here is that the assessment discarded the linear world and sent us to a model that can express the boundary, without training every option first.

## A classically easy dataset

Things look different with the breast cancer dataset. It has 30 features, more than the 16 qubits QSVC can simulate, and a logistic regression already gets most samples right.

In [4]:
X, y = datasets.load_breast_cancer(return_X_y=True, as_frame=True)

cancer = Project(X.assign(target=y), target="target")
cancer.preprocess()

assessment = cancer.assess()
assessment.report()

╭─ QML suitability ───────────────────────────────────────────────────────────────────────────────────────────────╮
│ 66.0 / 100   MEDIUM  Quantum models might help, with care                                                       │
│ 569 of 569 samples, 30 of 30 features, 2 classes, budget of 16 qubits                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─ Sub-scores ────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Fourier approx.     ━━━━━━━━━━━━━━━━━━━━╸━     94  average R² 0.781                                             │
│ Trainability        ━━━━━━━━━━━━━━━╸━━━━━━     72  28.0% of frequencies unreachable (ternary init)              │
│ Circuit depth       ━━━━━━━━━━━━━╺━━━━━━━━     60  fixed frequency depth ≈ 1344                                 │
│ Classical hardness  ━━━╸━━━━━━━━━━━━━━━━━━     17  logistic bal. acc. 0.966, tree gap -0.055                    │
│ Class balance       ━━━━━━━━━━━━━━━╸━━━━━━     71  ratio 0.594 (mild)                                           │
│ Dimensionality      ━━━━━━━━━━━━━━━━━━━━━━    100  ≥ 7 qubits needed                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
Raw table: 30 columns become 30 features                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Column                             ┃ Kind        ┃     Missing ┃     Unique ┃     Features ┃      Target assoc. ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ mean radius                        │ numeric     │          0% │        456 │            1 │               0.76 │
│ mean texture                       │ numeric     │          0% │        479 │            1 │               0.48 │
│ mean perimeter                     │ numeric     │          0% │        522 │            1 │               0.78 │
│ mean area                          │ numeric     │          0% │        539 │            1 │               0.76 │
│ mean smoothness                    │ numeric     │          0% │        474 │            1 │               0.39 │
│ mean compactness                   │ numeric     │          0% │        537 │            1 │               0.61 │
│ mean concavity                     │ numeric     │          0% │        537 │            1 │               0.78 │
│ mean concave points                │ numeric     │          0% │        542 │            1 │               0.83 │
│ mean symmetry                      │ numeric     │          0% │        432 │            1 │               0.35 │
│ mean fractal dimension             │ numeric     │          0% │        499 │            1 │               0.20 │
│ radius error                       │ numeric     │          0% │        540 │            1 │               0.63 │
│ texture error                      │ numeric     │          0% │        519 │            1 │               0.12 │
│ perimeter error                    │ numeric     │          0% │        533 │            1 │               0.57 │
│ area error                         │ numeric     │          0% │        528 │            1 │               0.58 │
│ smoothness error                   │ numeric     │          0% │        547 │            1 │               0.16 │
│ compactness error                  │ numeric     │          0% │        541 │            1 │               0.37 │
│ concavity error                    │ numeric     │          0% │        533 │            1 │               0.28 │
│ concave points error               │ numeric     │          0% │        507 │            1 │               0.48 │
│ symmetry error                     │ numeric     │    

The classical SVC comes first and the report points to the steps that could make the dataset friendlier to a quantum model. Let's follow one of them, keeping the 6 most informative features, and assess again for a budget of 6 qubits.

In [5]:
cancer.feature_selection(max_cols=6, seed=0)
cancer.assess(n_qubits=6)

╭─ QML suitability ────────────────────────────────────────────────────────────╮
│ 69.3 / 100   MEDIUM  Quantum models might help, with care                    │
│ 569 of 569 samples, 6 of 6 features, 2 classes, budget of 6 qubits           │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Sub-scores ─────────────────────────────────────────────────────────────────╮
│ Fourier approx.     ━━━━━━━━━━━━━━━━━━━━━╸    100  average R² 0.833          │
│ Trainability        ━━━━━━━━━━━━━━━━━╺━━━━     78  22.2% of frequencies      │
│                                                    unreachable (ternary      │
│                                                    init)                     │
│ Circuit depth       ━━━━━━━━━━━━━╺━━━━━━━━     60  fixed frequency depth ≈   │
│                                                    416                       │
│ Classical hardness  ━━━━╺━━━━━━━━━━━━━━━━━     19  logistic bal. acc. 0.949, │
│                                                    tree gap -0.028           │
│ Class balance       ━━━━━━━━━━━━━━━╸━━━━━━     71  ratio 0.594 (mild)        │
│ Dimensionality      ━━━━━━━━━━━━━━━━━━━━━━    100  ≥ 5 qubits needed         │
╰──────────────────────────────────────────────────────────────────────────────╯
Raw table: 30 columns become 30 features                                        
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Column               ┃ Kind    ┃ Missing ┃ Unique ┃ Features ┃ Target assoc. ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mean radius          │ numeric │      0% │    456 │        1 │          0.76 │
│ mean texture         │ numeric │      0% │    479 │        1 │          0.48 │
│ mean perimeter       │ numeric │      0% │    522 │        1 │          0.78 │
│ mean area            │ numeric │      0% │    539 │        1 │          0.76 │
│ mean smoothness      │ numeric │      0% │    474 │        1 │          0.39 │
│ mean compactness     │ numeric │      0% │    537 │        1 │          0.61 │
│ mean concavity       │ numeric │      0% │    537 │        1 │          0.78 │
│ mean concave points  │ numeric │      0% │    542 │        1 │          0.83 │
│ mean symmetry        │ numeric │      0% │    432 │        1 │          0.35 │
│ mean fractal         │ numeric │      0% │    499 │        1 │          0.20 │
│ dimension            │         │         │        │          │               │
│ radius error         │ numeric │      0% │    540 │        1 │          0.63 │
│ texture error        │ numeric │      0% │    519 │        1 │          0.12 │
│ perimeter error      │ numeric │      0% │    533 │        1 │          0.57 │
│ area error           │ numeric │      0% │    528 │        1 │          0.58 │
│ smoothness error     │ numeric │      0% │    547 │        1 │          0.16 │
│ compactness error    │ numeric │      0% │    541 │        1 │          0.37 │
│ concavity error      │ numeric │      0% │    533 │        1 │          0.28 │
│ concave points error │ numeric │      0% │    507 │        1 │          0.48 │
│ symmetry error       │ numeric │      0% │    498 │        1 │          0.20 │
│ fractal dimension    │ numeric │      0% │    545 │        1 │          0.18 │
│ error                │         │         │        │          │               │
│ worst radius         │ numeric │      0% │    457 │        1 │          0.84 │
│ worst texture        │ numeric │      0% │    511 │        1 │          0.48 │
│ worst perimeter      │ numeric │      0% │    514 │        1 │          0.85 │
│ worst area           │ numeric │      0% │    544 │        1 │          0.81 │
│ worst smoothness     │ numeric │      0% │    411 │        1 │          0.44 │
│ worst compactness    │ numeric │      0% │    529 │        1 │          0.62 │
│ worst concavity      │ numeric │      0% │    539 │        1 │          0.71 │
│ worst concave points │ numeri

Now quantum kernels fit in the budget, but the task is still easy for a linear model and the classical baseline keeps its place. That is a perfectly valid answer too: Falcondale is about adding quantum models to the toolbox when they make sense, not replacing what already works.

The ranking can be inspected programmatically.

In [6]:
for rec in cancer.assessment.recommendations_:
    print(f"{rec.name:<26} score {rec.score:5.1f}  qubits {rec.qubits}  {rec.reasons[0]}")

svc                        score  81.4  qubits 0  classical baseline, overall QML suitability is medium
qsvc (backend=qiskit)      score  79.7  qubits 6  78% of the frequencies within reach of a repeated encoding
qsvc (backend=pennylane)   score  65.7  qubits 6  39% of the frequencies within reach of a single rotation
qnn                        score  56.6  qubits 3  amplitude embedding fits 6 features in 3 qubits


## Outside a project

`QMLAssessor` is also available on its own. It follows the scikit-learn estimator API, works with numpy arrays or dataframes and handles multiclass problems. Here with the wine dataset and a budget of 8 qubits.

In [7]:
from falcondale import QMLAssessor

X, y = datasets.load_wine(return_X_y=True, as_frame=True)
assessor = QMLAssessor(n_qubits=8).fit(X, y)

assessor.score_, assessor.verdict_, assessor.best_model.name

(64.7, 'MEDIUM', 'svc')

In [8]:
assessor.sub_scores_

{'fourier': 69.3,
 'trainability': 86.4,
 'depth': 60.0,
 'difficulty': 18.3,
 'imbalance': 81.1,
 'dimensionality': 100.0}

Everything can be exported as plain Python types with `to_dict()`, handy to keep track of the assessments of different versions of a dataset.

In [9]:
import json

print(json.dumps(assessor.to_dict()["details"], indent=2))

{
  "informative_features": 13,
  "omega_max": 13,
  "frequencies": 29,
  "avg_fourier_r2": 0.5776,
  "unreachable_zero_pct": 41.4,
  "unreachable_ternary_pct": 13.6,
  "depth_estimate": 1469,
  "qubits_needed": 5,
  "lr_balanced_accuracy": 0.9765258215962441,
  "nonlinearity_gap": -0.03423815018169285
}
